In [4]:


import json
import os
import re
import sys
from rich import print as rprint

from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langfuse import get_client as get_langfuse_client
from langfuse.api.commons.errors.error import Error as LangfuseError
from langfuse.experiment import Evaluation

# 复用 lesson 0005-02 的 judge
from src.apps.fuse.scoring_demo import judge_helpfulness

from langfuse.api import Dataset
from langfuse._client.datasets import DatasetClient
from langfuse import Langfuse

load_dotenv()

DATASET_NAME = "movie-critic-eval-v1"
PROMPT_NAME = "movie-critic"


In [5]:
# 公共 LLM（task 和 judge 共用；实战 judge 用 gpt-4o-mini 等不同模型）
llm = ChatOpenAI(
    model="MiniMax-M3",
    api_key=os.getenv("MINIMAX_API_KEY"),
    base_url=os.getenv("MINIMAX_BASE_URL"),
    temperature=0.7,
)

langfuse:Langfuse = get_langfuse_client()



In [3]:

# 10 条样本：覆盖不同风格 + 不同电影
DATASET_ITEMS = [
    {"id": "q1",  "input": {"criticlevel": "harsh",  "movie": "Dune 2"}},
    {"id": "q2",  "input": {"criticlevel": "kind",   "movie": "Barbie"}},
    {"id": "q3",  "input": {"criticlevel": "neutral", "movie": "Oppenheimer"}},
    {"id": "q4",  "input": {"criticlevel": "harsh",  "movie": "The Marvels"}},
    {"id": "q5",  "input": {"criticlevel": "kind",   "movie": "Elemental"}},
    {"id": "q6",  "input": {"criticlevel": "neutral", "movie": "Killers of the Flower Moon"}},
    {"id": "q7",  "input": {"criticlevel": "harsh",  "movie": "Madame Web"}},
    {"id": "q8",  "input": {"criticlevel": "kind",   "movie": "Wonka"}},
    {"id": "q9",  "input": {"criticlevel": "neutral", "movie": "Past Lives"}},
    {"id": "q10", "input": {"criticlevel": "harsh",  "movie": "Morbius"}},
]


In [6]:


# ---------- 1. 建 Dataset（幂等） ----------

def ensure_dataset() -> None:

    try:
        ds_client:DatasetClient = langfuse.get_dataset(DATASET_NAME)
        rprint(f"✓ dataset '{ds_client}' already exists")
    except Exception as e:
        rprint(f"❌ dataset '{DATASET_NAME}' not found: {e}")
        dataset:Dataset = langfuse.create_dataset(
            name=DATASET_NAME,
            description="Evaluate movie-critic prompt across 10 sample films × 3 critic levels",
        )
        rprint(f"✓ created dataset '{dataset}")

        for item in DATASET_ITEMS:
            langfuse.create_dataset_item(
                dataset_name=DATASET_NAME,
                id=item["id"],
                input=item["input"],
                expected_output=None,  # LLM 输出不确定，留空用 LLM Judge 评
            )
        ds_client:langfuse.get_dataset(DATASET_NAME)
        rprint(f'added {len(ds_client.items)} data set item to dataset {DATASET_NAME}')

ensure_dataset()

# def ensure_dataset() -> None:
#     """幂等创建 dataset + 10 条 item。"""
#     lf = get_client()
#     try:
#         lf.get_dataset(DATASET_NAME)
#         print(f"✓ dataset '{DATASET_NAME}' already exists")
#     except Exception:
#         # NotFoundError / 任何 API 错误都视作"不存在" → 创建
#         # （Langfuse SDK 4.x 里 Error 和 NotFoundError 是兄弟节点，单独 except 接不到）
#         lf.create_dataset(
#             name=DATASET_NAME,
#             description="Evaluate movie-critic prompt across 10 sample films × 3 critic levels",
#         )
#         print(f"✓ created dataset '{DATASET_NAME}'")
#
#     # 批量 upsert items
#     for it in DATASET_ITEMS:
#         lf.create_dataset_item(
#             dataset_name=DATASET_NAME,
#             id=it["id"],
#             input=it["input"],
#             expected_output=None,  # LLM 输出不确定，留空用 LLM Judge 评
#         )
#     print(f"✓ {len(DATASET_ITEMS)} items upserted")



✓ dataset '<langfuse._client.datasets.DatasetClient object at 0x10c445c50>' already exists

In [7]:

def make_task(label: str):
    """返回一个 task closure：拿指定 label 的 prompt 跑 input → output。

    v4 SDK 调 task 时用 keyword: task(item=ExperimentItem)，
    ExperimentItem 是 TypedDict 含 input / expected_output / metadata。
    """
    prompt_obj = langfuse.get_prompt(PROMPT_NAME, label=label)

    def task(item, **kwargs) -> str:
        # item 是 TypedDict，input 字段是用户存的 dict
        item_input = item["input"] if isinstance(item, dict) else item.input
        compiled = prompt_obj.compile(**item_input)
        resp = llm.invoke(
            [
                {"role": "system", "content": compiled},
                {"role": "user", "content": f"评论一下《{item_input['movie']}》"},
            ]
        )
        return resp.content

    return task, prompt_obj.version

for item in ["production","latest"]:
    result = make_task(label=item)
    rprint(result)


(<function make_task.<locals>.task at 0x10c44a5c0>, 1)

(<function make_task.<locals>.task at 0x10c6f6c00>, 1)